<h1><font color='blue'>Análise de Sentimentos e Sistemas de Recomendação</font></h1>
<h2><b>Prática U1-T2-V2.2-Abordagens baseadas em léxico</b></h2>


---

## Preparação

### Carregando o dataset

In [1]:
import pandas as pd
df = pd.read_csv('https://drive.google.com/uc?export=download&id=1M7VtNfEh7K3gqa4OiWiayPkg8q6x8fXk', encoding='utf-8', low_memory=False)
df.head()

,frase_pt,frase_en,sentimento
0,O atendimento ao cliente foi excepcional e res...,The customer service was exceptional and solve...,Positivo
1,"Este é, sem dúvida, o melhor café que já prove...","This is, without a doubt, the best coffee I've...",Positivo
2,O filme tem uma fotografia incrível e uma tril...,The movie has incredible cinematography and a ...,Positivo
3,Estou muito satisfeito com a qualidade do prod...,I am very satisfied with the product's quality...,Positivo
4,A equipe foi extremamente simpática e prestati...,The staff was extremely friendly and helpful t...,Positivo


In [2]:
df['sentimento'].value_counts()

,count
sentimento,
Positivo,10
Negativo,10
Neutro,10


### Pré-processamento do texto

In [3]:
import nltk

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

True

In [4]:
import re
from nltk.tokenize import word_tokenize
import unicodedata

def preprocess(text):
    text = text.lower()
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode("utf-8")
    text = re.sub(r'[^a-z0-9\s]', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    tokens = word_tokenize(text)
    return tokens

preprocess('O trânsito      HOJE estava    $$$ %  &  #simplesmente insuportável.')

['o', 'transito', 'hoje', 'estava', 'simplesmente', 'insuportavel']

## Abordagem usando VADER

### 📘 O léxico VADER (vader_lexicon)

#### O que é o VADER?
O **VADER (Valence Aware Dictionary and sEntiment Reasoner)** é um dicionário léxico desenvolvido especificamente para **análise de sentimentos em textos curtos e informais**, como comentários em redes sociais, avaliações online e mensagens instantâneas.

Diferente de léxicos tradicionais, o VADER foi criado levando em conta **a forma como as pessoas realmente escrevem na internet**, incluindo gírias, pontuação e emojis.

---

#### 🎯 Principais Características

- Léxico **pré-definido** com palavras associadas a sentimentos
- Cada termo possui um **valor de polaridade** (positivo ou negativo)
- Sensível a **intensidade emocional**
- Especialmente eficaz para textos curtos
- Muito utilizado como **baseline** em projetos de PLN

---

#### 🧮 Como o vader_lexicon funciona

O VADER atribui uma **pontuação de sentimento** às palavras e ajusta essa pontuação considerando regras heurísticas, como:

- **Intensificadores**  
  Ex.: *"muito bom"* → sentimento mais forte

- **Negação**  
  Ex.: *"não gostei"* → inversão da polaridade

- **Pontuação**  
  Ex.: *"excelente!!!"* → maior intensidade

- **Uso de MAIÚSCULAS**  
  Ex.: *"ÓTIMO"* → ênfase emocional

- **Emojis e emoticons**  
  Ex.: 😄 😡

---

#### 📊 Saída do VADER

O VADER retorna quatro métricas principais:

- **pos** → proporção de sentimento positivo
- **neg** → proporção de sentimento negativo
- **neu** → proporção neutra
- **compoun**


In [5]:
nltk.download('vader_lexicon')


[nltk_data] Downloading package vader_lexicon to /root/nltk_data...


True

In [6]:
from nltk.sentiment.vader import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
analyzer.polarity_scores("I really love this movie. It's fantastic!")

{'neg': 0.0, 'neu': 0.323, 'pos': 0.677, 'compound': 0.855}

In [7]:
def retorna_sentimento(vader_scores):
    pos = vader_scores.get("pos", 0)
    neg = vader_scores.get("neg", 0)
    neu = vader_scores.get("neu", 0)

    if pos > neg and pos > neu:
        return "Positivo"
    elif neg > pos and neg > neu:
        return "Negativo"
    else:
        return "Neutro"

In [8]:
frases = df['frase_en'].values.tolist()
frases[:5]

['The customer service was exceptional and solved my problem quickly.',
 "This is, without a doubt, the best coffee I've ever tasted in the city!",
 'The movie has incredible cinematography and a moving soundtrack.',
 "I am very satisfied with the product's quality, it exceeded my expectations.",
 'The staff was extremely friendly and helpful throughout my entire visit.']

In [9]:
res = analyzer.polarity_scores('The customer service was exceptional and solved my problem quickly.')
print(res)
retorna_sentimento(res)

{'neg': 0.211, 'neu': 0.625, 'pos': 0.164, 'compound': -0.1531}


'Neutro'

In [10]:
# Criar uma instância do analisador de sentimento
analyzer = SentimentIntensityAnalyzer()

# Analisar o sentimento de cada frase
y_pred = []
for frase, sentimento in df[['frase_en','sentimento']].values.tolist():
    res = analyzer.polarity_scores(frase)
    # Determinar o sentimento geral com base na pontuação composta
    retorno = retorna_sentimento(res)
    y_pred.append(retorno)
    print(f'Frase: {frase:<90} - Sentimento: pred "{retorno}" - real "{sentimento}" \n\t{res}')


Frase: The customer service was exceptional and solved my problem quickly.                        - Sentimento: pred "Neutro" - real "Positivo" 
	{'neg': 0.211, 'neu': 0.625, 'pos': 0.164, 'compound': -0.1531}
Frase: This is, without a doubt, the best coffee I've ever tasted in the city!                    - Sentimento: pred "Neutro" - real "Positivo" 
	{'neg': 0.218, 'neu': 0.656, 'pos': 0.126, 'compound': -0.3716}
Frase: The movie has incredible cinematography and a moving soundtrack.                           - Sentimento: pred "Neutro" - real "Positivo" 
	{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Frase: I am very satisfied with the product's quality, it exceeded my expectations.               - Sentimento: pred "Neutro" - real "Positivo" 
	{'neg': 0.0, 'neu': 0.764, 'pos': 0.236, 'compound': 0.4754}
Frase: The staff was extremely friendly and helpful throughout my entire visit.                   - Sentimento: pred "Neutro" - real "Positivo" 
	{'neg': 0.0, 'neu': 0.579, 

Ver a taxa de acerto (acurácia)

In [11]:
from sklearn.metrics import accuracy_score

y_true = df['sentimento'].values.tolist()
accuracy_score(y_true, y_pred)

0.3333333333333333

In [12]:
def retorna_sentimento(vader_scores, limiar_compound=0.05, limiar_neutro=0.8):
    neg = vader_scores.get("neg", 0)
    neu = vader_scores.get("neu", 0)
    pos = vader_scores.get("pos", 0)
    compound = vader_scores.get("compound", 0)

    # Predominância de neutralidade
    if neu >= limiar_neutro:
        return "Neutro"
    # Decisão baseada no compound
    elif compound >= limiar_compound and pos > neg:
        return "Positivo"
    elif compound <= -limiar_compound and neg > pos:
        return "Negativo"
    else:
        return "Neutro"

In [13]:
# Criar uma instância do analisador de sentimento
analyzer = SentimentIntensityAnalyzer()

# Analisar o sentimento de cada frase
y_pred = []
for frase, sentimento in df[['frase_en','sentimento']].values.tolist():
    res = analyzer.polarity_scores(frase)
    # Determinar o sentimento geral com base na pontuação composta
    retorno = retorna_sentimento(res)
    y_pred.append(retorno)
    print(f'Frase: {frase:<90} - Sentimento: pred "{retorno}" - real "{sentimento}" \n\t{res}')


Frase: The customer service was exceptional and solved my problem quickly.                        - Sentimento: pred "Negativo" - real "Positivo" 
	{'neg': 0.211, 'neu': 0.625, 'pos': 0.164, 'compound': -0.1531}
Frase: This is, without a doubt, the best coffee I've ever tasted in the city!                    - Sentimento: pred "Negativo" - real "Positivo" 
	{'neg': 0.218, 'neu': 0.656, 'pos': 0.126, 'compound': -0.3716}
Frase: The movie has incredible cinematography and a moving soundtrack.                           - Sentimento: pred "Neutro" - real "Positivo" 
	{'neg': 0.0, 'neu': 1.0, 'pos': 0.0, 'compound': 0.0}
Frase: I am very satisfied with the product's quality, it exceeded my expectations.               - Sentimento: pred "Positivo" - real "Positivo" 
	{'neg': 0.0, 'neu': 0.764, 'pos': 0.236, 'compound': 0.4754}
Frase: The staff was extremely friendly and helpful throughout my entire visit.                   - Sentimento: pred "Positivo" - real "Positivo" 
	{'neg': 0.0, 'neu':

In [14]:
accuracy_score(y_true, y_pred)

0.7333333333333333

### 📘 LeIA (Lexicon-based Sentiment Analysis for Portuguese)

A **LeIA** é uma biblioteca de **análise de sentimentos baseada em léxico**, desenvolvida especificamente para o **português**, com foco no **português brasileiro**. Seu objetivo é identificar a **polaridade emocional** de textos curtos, como comentários, avaliações e postagens em redes sociais.

Inspirada no **VADER**, a LeIA combina um **dicionário de palavras com escores de sentimento** e um conjunto de **regras linguísticas** que modelam fenômenos comuns da linguagem natural, como **negação**, **intensificadores**, **pontuação**, **uso de emojis** e expressões coloquiais do português.

A biblioteca retorna um **dicionário de pontuações**, contendo os valores:
- **pos**: intensidade do sentimento positivo
- **neg**: intensidade do sentimento negativo
- **neu**: intensidade do sentimento neutro
- **compound**: escore global normalizado do sentimento

A LeIA é amplamente utilizada em **contextos educacionais**, **monitoramento de opiniões**, **análise de feedbacks** e **prototipagem rápida**, sendo uma solução eficiente quando não há grandes volumes de dados rotulados para treinamento de modelos de machine learning.


In [15]:
!pip install -q leia-br

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.0/130.0 kB 4.0 MB/s eta 0:00:00


In [16]:
from LeIA import SentimentIntensityAnalyzer

analyzer = SentimentIntensityAnalyzer()
analyzer.polarity_scores('O atendimento ao cliente foi excepcional e resolveu meu problema rapidamente.')

{'neg': 0.181, 'neu': 0.403, 'pos': 0.416, 'compound': 0.5423}

In [17]:
# Criar uma instância do analisador de sentimento
analyzer = SentimentIntensityAnalyzer()

# Analisar o sentimento de cada frase
y_pred = []
for frase, sentimento in df[['frase_pt','sentimento']].values.tolist():
    res = analyzer.polarity_scores(frase)
    # Determinar o sentimento geral com base na pontuação composta
    retorno = retorna_sentimento(res)
    y_pred.append(retorno)
    print(f'Frase: {frase:<90} - Sentimento: pred "{retorno}" - real "{sentimento}" \n\t{res}')


Frase: O atendimento ao cliente foi excepcional e resolveu meu problema rapidamente.              - Sentimento: pred "Positivo" - real "Positivo" 
	{'neg': 0.181, 'neu': 0.403, 'pos': 0.416, 'compound': 0.5423}
Frase: Este é, sem dúvida, o melhor café que já provei na cidade!                                 - Sentimento: pred "Positivo" - real "Positivo" 
	{'neg': 0.153, 'neu': 0.625, 'pos': 0.222, 'compound': 0.2481}
Frase: O filme tem uma fotografia incrível e uma trilha sonora emocionante.                       - Sentimento: pred "Positivo" - real "Positivo" 
	{'neg': 0.0, 'neu': 0.721, 'pos': 0.279, 'compound': 0.4767}
Frase: Estou muito satisfeito com a qualidade do produto, superou minhas expectativas.            - Sentimento: pred "Positivo" - real "Positivo" 
	{'neg': 0.0, 'neu': 0.763, 'pos': 0.237, 'compound': 0.4215}
Frase: A equipe foi extremamente simpática e prestativa durante toda a minha visita.              - Sentimento: pred "Neutro" - real "Positivo" 
	{'neg': 0.0, '

In [18]:
accuracy_score(y_true, y_pred)

0.7

## Abordagem usando TextBlob

### 📘 TextBlob

O **TextBlob** é uma biblioteca de **Processamento de Linguagem Natural (PLN)** em Python que oferece funcionalidades simples e acessíveis para tarefas como **análise de sentimentos**, **tokenização**, **extração de n-gramas**, **tradução** e **classificação de texto**. Sua proposta principal é facilitar o uso de técnicas de PLN, especialmente em contextos educacionais e de prototipagem.

Na análise de sentimentos, o TextBlob utiliza uma abordagem **baseada em léxico**, apoiada no léxico **Pattern**, que associa palavras a valores de **polaridade** e **subjetividade**. A polaridade indica o quão positiva ou negativa é uma opinião, enquanto a subjetividade mede o grau de opinião versus fato presente no texto.

A saída do TextBlob é composta principalmente por dois valores:
- **polarity**: varia de -1 (negativo) a +1 (positivo)
- **subjectivity**: varia de 0 (objetivo) a 1 (subjetivo)

O TextBlob é amplamente utilizado para **análises rápidas**, **demonstrações em sala de aula** e **projetos iniciais**, sendo uma ferramenta intuitiva para introduzir conceitos fundamentais de análise de sentimentos, embora apresente limitações em textos complexos, irônicos ou fora do domínio do inglês.


In [19]:
from textblob import TextBlob

res = TextBlob('The customer service was exceptional and solved my problem quickly.')
res.sentiment

Sentiment(polarity=0.5, subjectivity=0.75)

In [20]:
def retorna_sentimento(sentiment):
    polarity = sentiment.polarity

    if polarity > 0:
        return "Positivo"
    elif polarity < 0:
        return "Negativo"
    else:
        return "Neutro"

retorna_sentimento(res.sentiment)

'Positivo'

In [21]:
y_pred = []
for frase, sentimento in df[['frase_en','sentimento']].values.tolist():
    res = TextBlob(frase)
    # Determinar o sentimento geral com base na pontuação composta
    retorno = retorna_sentimento(res.sentiment)
    y_pred.append(retorno)
    print(f'Frase: {frase:<90} - Sentimento: pred "{retorno}" - real "{sentimento}" \n\t{res.sentiment}')


Frase: The customer service was exceptional and solved my problem quickly.                        - Sentimento: pred "Positivo" - real "Positivo" 
	Sentiment(polarity=0.5, subjectivity=0.75)
Frase: This is, without a doubt, the best coffee I've ever tasted in the city!                    - Sentimento: pred "Positivo" - real "Positivo" 
	Sentiment(polarity=1.0, subjectivity=0.3)
Frase: The movie has incredible cinematography and a moving soundtrack.                           - Sentimento: pred "Positivo" - real "Positivo" 
	Sentiment(polarity=0.9, subjectivity=0.9)
Frase: I am very satisfied with the product's quality, it exceeded my expectations.               - Sentimento: pred "Positivo" - real "Positivo" 
	Sentiment(polarity=0.65, subjectivity=1.0)
Frase: The staff was extremely friendly and helpful throughout my entire visit.                   - Sentimento: pred "Positivo" - real "Positivo" 
	Sentiment(polarity=0.1875, subjectivity=0.5625)
Frase: What a wonderful experience! I will

In [22]:
accuracy_score(y_true, y_pred)

0.8333333333333334